In [1]:
from langgraph.types import Send
from langchain_core.messages import content
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage,ToolMessage,AIMessage
from langgraph.graph import StateGraph, START, END
from app.LLM.llm import prompt_enhancer_llm, planner_llm, researcher_llm, worker_llm,llm
from typing import TypedDict, Annotated
from app.prompts.planner import planner_prompt
from app.schema.planner_schema import PlannerSchema
from app.prompts.worker import worker_prompt
from app.schema.worker_schema import WorkerSchema
from app.tools.web_search import web_search
from langgraph.prebuilt import ToolNode, tools_condition
from app.prompts.promptEnhancer import prompt_enhancer_prompt
from app.schema.prompt_enhancer import PromptEnhancerSchema
from app.prompts.researcher import researcher_prompt
from langgraph.graph.message import add_messages
import operator
from pydantic import BaseModel, Field


In [2]:
worker_llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001B9A41E0050>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001B9A41E0ED0>, model_name='meta-llama/llama-4-scout-17b-16e-instruct', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:

class ResearcherState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    section:dict


In [4]:
def researcher(state: ResearcherState):
    messages = state["messages"]

    formatted = []
    for msg in messages:
        # truncate huge tool outputs
        if isinstance(msg, ToolMessage):
            content = str(msg.content)
            formatted.append(
                ToolMessage(
                    content=content[:500] + "...",  # truncate
                    tool_call_id=msg.tool_call_id,
                )
            )
        elif isinstance(msg, HumanMessage):
            formatted.append(msg)
        # optionally trim long ai reasoning
        elif isinstance(msg, AIMessage):
            formatted.append(
                AIMessage(content=str(msg.content)[:1500], tool_calls=msg.tool_calls)
            )
    researcher = researcher_llm.bind_tools([web_search]).invoke(
        [SystemMessage(content=researcher_prompt), *formatted]
    )
    print("======================\n",formatted)
    return {"messages": [researcher]}

In [5]:
def writer(state:ResearcherState):
    messages=state['messages']
    print("messages")
    research=[]
    for i in messages:
        if( isinstance(i,HumanMessage)):
            research.append(HumanMessage(content=i.content))
        elif(isinstance(i,ToolMessage)):
            research.append(HumanMessage(content=f"Research findings : \n{i.content}")) 
    
    worker = llm.with_structured_output(WorkerSchema).invoke(
        [SystemMessage(content=worker_prompt), *research]
    )
    return {"section":worker}

In [6]:
llm

ChatFireworks(profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2025-08-05', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<fireworks.resources.chat.completions.CompletionsResource object at 0x000001B9A3F56B10>, async_client=<fireworks.resources.chat.completions.AsyncCompletionsResource object at 0x000001B9A3FFF490>, model_name='accounts/fireworks/models/gpt-oss-120b', model_kwargs={}, fireworks_api_key=SecretStr('**********'))

In [7]:
messages = [
    HumanMessage(
        content=f"""
Research Topic : {"Large Language Models (LLMs)"}

Research Section : {"Introduction and Core Foundations"}

Research Questions : {
            [
                "What are Large Language Models?",
                "How do LLMs function?", 
            ]
        }

Key Topics : {
            [
                "Large Language Models",
                "Transformer Architecture", 
            ]
        }

Research Brief : {
            "A foundational educational study on Large Language Models, focusing on their architecture , and working mechanisms."
        }
"""
    )
]

In [8]:

tools=[web_search]
toolnode=ToolNode(tools)
graph=StateGraph(ResearcherState)
graph.add_node("researcher",researcher)
graph.add_node("tools",toolnode)
graph.add_node("writer",writer)

graph.add_edge(START,"researcher")
graph.add_conditional_edges("researcher",tools_condition,{"__end__":"writer","tools":"tools"})
graph.add_edge("tools","researcher")
graph.add_edge("writer",END)

wf=graph.compile()



In [9]:
for i in wf.stream({"messages":messages},stream_mode="updates"):
   print(i,end="" )
   print("----------------------------------------------------------------------------------------")

 [HumanMessage(content="\nResearch Topic : Large Language Models (LLMs)\n\nResearch Section : Introduction and Core Foundations\n\nResearch Questions : ['What are Large Language Models?', 'How do LLMs function?']\n\nKey Topics : ['Large Language Models', 'Transformer Architecture']\n\nResearch Brief : A foundational educational study on Large Language Models, focusing on their architecture , and working mechanisms.\n", additional_kwargs={}, response_metadata={}, id='0bfb05db-b89f-4196-8875-420defd20fc8')]
{'researcher': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'dggqsczaj', 'function': {'arguments': '{"query":"What are Large Language Models and how do they function"}', 'name': 'web_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 1031, 'total_tokens': 1069, 'completion_time': 0.088260408, 'completion_tokens_details': None, 'prompt_time': 0.04112803, 'prompt_tokens_details': None, 'queue_time

In [10]:
open("report.md","w",encoding="utf-8").write(res['section'].content)

NameError: name 'res' is not defined

In [ ]:
llm.invoke("Hi")
